In [1]:
import sqlite3
import pandas as pd
import random
import os
from datetime import datetime
 
random.seed(99)

In [2]:
LEGACY_DB   = "data/legacy.db"
MIGRATED_DB = "data/migrated.db"
EXPORT_DIR  = "data/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

In [3]:
def convert_date(date_str: str) -> str:
    """Convert DD/MM/YYYY → ISO 8601 (YYYY-MM-DD). Returns None if malformed."""
    try:
        return datetime.strptime(date_str.strip(), "%d/%m/%Y").strftime("%Y-%m-%d")
    except (ValueError, AttributeError):
        return None
 
# ── Inject errors ─────────────────────────────────────────────────────────────
def maybe_drop(row_index: int, rate: float = 0.02) -> bool:
    """Returns True if this record should be dropped (simulating ETL failure)."""
    return random.random() < rate
 
def maybe_corrupt_amount(amount: float, rate: float = 0.01) -> float:
    """Introduces rounding error on ~1% of records."""
    if random.random() < rate:
        return round(amount + random.uniform(-15.0, 15.0), 2)
    return amount
 
def maybe_null_field(value, rate: float = 0.005):
    """Nullifies a value on ~0.5% of records."""
    if random.random() < rate:
        return None
    return value

In [4]:
def create_migrated_schema(conn: sqlite3.Connection):
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id       TEXT PRIMARY KEY,
            first_name        TEXT,
            last_name         TEXT,
            email             TEXT,
            phone             TEXT,
            address           TEXT,
            region            TEXT,
            fuel_type         TEXT,
            tariff_plan       TEXT,
            account_open_date TEXT,   -- ISO 8601
            payment_method    TEXT,
            is_active         INTEGER,
            migrated_at       TEXT DEFAULT (datetime('now'))
        );
 
        CREATE TABLE IF NOT EXISTS accounts (
            account_id        TEXT PRIMARY KEY,
            customer_id       TEXT,
            account_type      TEXT,
            credit_limit      REAL,
            current_balance   REAL,
            last_payment_date TEXT,   -- ISO 8601
            migrated_at       TEXT DEFAULT (datetime('now'))
        );
 
        CREATE TABLE IF NOT EXISTS billing_cycles (
            cycle_id          TEXT PRIMARY KEY,
            account_id        TEXT,
            cycle_start       TEXT,   -- ISO 8601
            cycle_end         TEXT,   -- ISO 8601
            units_consumed    REAL,
            unit_rate         REAL,
            standing_charge   REAL,
            vat_rate          REAL,
            amount_due        REAL,
            due_date          TEXT,   -- ISO 8601
            status            TEXT,
            migrated_at       TEXT DEFAULT (datetime('now'))
        );
 
        CREATE TABLE IF NOT EXISTS payments (
            payment_id        TEXT PRIMARY KEY,
            account_id        TEXT,
            cycle_id          TEXT,
            payment_date      TEXT,   -- ISO 8601
            amount_paid       REAL,
            payment_method    TEXT,
            status            TEXT,
            migrated_at       TEXT DEFAULT (datetime('now'))
        );
    """)
    conn.commit()

In [5]:
def migrate_customers(src: sqlite3.Connection, dst: sqlite3.Connection) -> dict:
    df = pd.read_sql("SELECT * FROM customers", src)
    total = len(df)
    migrated, dropped = [], 0
 
    for i, row in df.iterrows():
        if maybe_drop(i, rate=0.02):
            dropped += 1
            continue
        migrated.append((
            row["customer_id"],
            maybe_null_field(row["first_name"]),
            row["last_name"],
            row["email"],
            row["phone"],
            row["address"],
            row["region"],
            row["fuel_type"],
            row["tariff_plan"],
            convert_date(row["account_open_date"]),  # format transform
            row["payment_method"],
            row["is_active"]
        ))
 
    dst.executemany(
        "INSERT OR IGNORE INTO customers VALUES (?,?,?,?,?,?,?,?,?,?,?,?,datetime('now'))",
        migrated
    )
    dst.commit()
    return {"table": "customers", "source_rows": total, "migrated_rows": len(migrated), "dropped": dropped}

In [6]:
def migrate_accounts(src: sqlite3.Connection, dst: sqlite3.Connection) -> dict:
    df = pd.read_sql("SELECT * FROM accounts", src)
    total = len(df)
    migrated, dropped = [], 0
 
    for i, row in df.iterrows():
        if maybe_drop(i, rate=0.02):
            dropped += 1
            continue
        migrated.append((
            row["account_id"],
            row["customer_id"],
            row["account_type"],
            row["credit_limit"],
            maybe_corrupt_amount(row["current_balance"]),  # balance corruption
            convert_date(row["last_payment_date"])
        ))
 
    dst.executemany(
        "INSERT OR IGNORE INTO accounts VALUES (?,?,?,?,?,?,datetime('now'))",
        migrated
    )
    dst.commit()
    return {"table": "accounts", "source_rows": total, "migrated_rows": len(migrated), "dropped": dropped}

In [7]:
def migrate_billing_cycles(src: sqlite3.Connection, dst: sqlite3.Connection) -> dict:
    df = pd.read_sql("SELECT * FROM billing_cycles", src)
    total = len(df)
    migrated, dropped = [], 0
 
    for i, row in df.iterrows():
        if maybe_drop(i, rate=0.02):
            dropped += 1
            continue
        migrated.append((
            row["cycle_id"],
            row["account_id"],
            convert_date(row["cycle_start"]),
            convert_date(row["cycle_end"]),
            row["units_consumed"],
            row["unit_rate"],
            row["standing_charge"],
            row["vat_rate"],
            maybe_corrupt_amount(row["amount_due"]),  # amount corruption
            convert_date(row["due_date"]),
            row["status"]
        ))
 
    # Batch insert
    batch = 10_000
    for k in range(0, len(migrated), batch):
        dst.executemany(
            "INSERT OR IGNORE INTO billing_cycles VALUES (?,?,?,?,?,?,?,?,?,?,?,datetime('now'))",
            migrated[k:k+batch]
        )
    dst.commit()
    return {"table": "billing_cycles", "source_rows": total, "migrated_rows": len(migrated), "dropped": dropped}

In [8]:
def migrate_payments(src: sqlite3.Connection, dst: sqlite3.Connection) -> dict:
    df = pd.read_sql("SELECT * FROM payments", src)
    total = len(df)
    migrated, dropped = [], 0
 
    for i, row in df.iterrows():
        if maybe_drop(i, rate=0.02):
            dropped += 1
            continue
        migrated.append((
            row["payment_id"],
            row["account_id"],
            row["cycle_id"],
            convert_date(row["payment_date"]),
            maybe_corrupt_amount(row["amount_paid"]),
            row["payment_method"],
            row["status"]
        ))
 
    batch = 10_000
    for k in range(0, len(migrated), batch):
        dst.executemany(
            "INSERT OR IGNORE INTO payments VALUES (?,?,?,?,?,?,?,datetime('now'))",
            migrated[k:k+batch]
        )
    dst.commit()
    return {"table": "payments", "source_rows": total, "migrated_rows": len(migrated), "dropped": dropped}

In [9]:
if __name__ == "__main__":
    src_conn = sqlite3.connect(LEGACY_DB)
    dst_conn = sqlite3.connect(MIGRATED_DB)
 
    create_migrated_schema(dst_conn)
 
    results = []
    print("Migrating customers...    ", end="", flush=True)
    results.append(migrate_customers(src_conn, dst_conn));    print("done")
 
    print("Migrating accounts...     ", end="", flush=True)
    results.append(migrate_accounts(src_conn, dst_conn));     print("done")
 
    print("Migrating billing cycles...", end="", flush=True)
    results.append(migrate_billing_cycles(src_conn, dst_conn)); print("done")
 
    print("Migrating payments...     ", end="", flush=True)
    results.append(migrate_payments(src_conn, dst_conn));     print("done")
 
    df_results = pd.DataFrame(results)
    df_results["drop_rate_pct"] = (df_results["dropped"] / df_results["source_rows"] * 100).round(2)
    print("\n── Migration Summary ─────────────────────────────────")
    print(df_results.to_string(index=False))
 
    df_results.to_csv(f"{EXPORT_DIR}/migration_summary.csv", index=False)
    print(f"\n✅ Migration complete. Summary → data/exports/migration_summary.csv")
 
    src_conn.close()
    dst_conn.close()

Migrating customers...    done
Migrating accounts...     done
Migrating billing cycles...done
Migrating payments...     done

── Migration Summary ─────────────────────────────────
         table  source_rows  migrated_rows  dropped  drop_rate_pct
     customers        50000          48998     1002           2.00
      accounts        50000          49009      991           1.98
billing_cycles       500000         490081     9919           1.98
      payments       449773         440608     9165           2.04

✅ Migration complete. Summary → data/exports/migration_summary.csv
